# 01: Business Understanding
**Credit Risk Prediction | Kaggle Competition**

Notebook ini berisi pemahaman bisnis sebelum kita masuk ke tahap teknis.
Tujuannya adalah memastikan kita tahu apa yang ingin dicapai, mengapa itu penting,
dan bagaimana kita akan mengukur keberhasilan solusi yang dibangun.

## 1. Ringkasan Studi Kasus

Kompetisi ini mensimulasikan kondisi nyata sebuah perusahaan fintech yang menyediakan
pinjaman kepada masyarakat. Setiap hari perusahaan menerima ratusan hingga ribuan
pengajuan pinjaman baru.

Tantangan utamanya adalah perusahaan harus memutuskan menyetujui atau menolak
pengajuan pada saat itu juga, tanpa bisa menunggu 12 bulan untuk melihat apakah
nasabah sungguh bermasalah dalam membayar.

Di sinilah Machine Learning berperan. Model dilatih menggunakan data historis
pengajuan yang sudah diketahui hasilnya, lalu digunakan untuk memprediksi
probabilitas risiko pada pengajuan baru.

### Definisi Target

Target dalam kompetisi ini bersifat biner:

| Nilai Target | Makna |
|---|---|
| `0` | Pinjaman tidak pernah mencapai 90 DPD dalam 12 bulan kalender setelah pinjaman dimulai |
| `1` | Pinjaman mencapai minimal 90 DPD setidaknya sekali dalam periode 12 bulan tersebut |

DPD atau Days Past Due adalah jumlah hari keterlambatan pembayaran dari tanggal jatuh tempo.
Ambang 90 hari dipilih karena mencerminkan kesulitan pembayaran yang serius,
bukan sekadar keterlambatan kecil yang masih umum terjadi.

### Sumber Data yang Tersedia

Dataset terdiri dari 8 file yang setiap file memberikan sudut pandang berbeda
terhadap profil nasabah:

| File | Isi Utama |
|---|---|
| `train_applications.csv` | Data pengajuan dengan label target untuk training |
| `test_applications.csv` | Data pengajuan tanpa label untuk prediksi final |
| `customer_profiles.csv` | Profil demografis dan pekerjaan nasabah |
| `previous_applications.csv` | Riwayat pengajuan pinjaman sebelumnya |
| `repayment_history.csv` | Catatan cicilan dan keterlambatan historis |
| `account_behavior.csv` | Ringkasan aktivitas rekening per bulan |
| `bureau_observations.csv` | Laporan biro kredit eksternal |
| `ecommerce_transactions.csv` | Riwayat transaksi belanja online |

### Output yang Diminta

Untuk setiap `application_id` di data test, model harus menghasilkan satu nilai
`default_probability` antara 0 dan 1. Nilai ini merepresentasikan estimasi risiko,
bukan label prediksi langsung.

## 2. Pertanyaan Bisnis

Tiga pertanyaan berikut dirancang untuk mencakup aspek yang berbeda dari
tujuan kompetisi ini secara menyeluruh.

### Pertanyaan Bisnis 1
**Faktor apa saja dari profil nasabah dan pengajuan pinjaman yang paling
berkaitan dengan risiko gagal bayar serius (90 DPD)?**

#### Konteks
Perusahaan memiliki banyak informasi saat nasabah mengajukan pinjaman:
pendapatan bulanan, jenis pekerjaan, jumlah tanggungan, tujuan pinjaman,
besarnya cicilan yang diusulkan, dan sebagainya.
Tidak semua informasi tersebut sama pentingnya bagi model.

#### Yang Ingin Dijawab
Apakah nasabah dengan beban cicilan yang tinggi relatif terhadap pendapatannya
lebih sering mengalami 90 DPD? Apakah jenis pekerjaan atau masa kerja berkaitan
dengan risiko? Apakah tujuan pinjaman tertentu memiliki pola risiko yang berbeda?

#### Relevansi Bisnis
Memahami seluruh faktor ini membantu kita membangun fitur yang punya alasan
bisnis yang kuat, bukan sekadar menambah kolom demi jumlah fitur.
Fitur yang berakar dari pemahaman bisnis juga lebih stabil saat digunakan
pada data baru.

#### Implikasi untuk Model
Jawaban dari pertanyaan ini akan memandu proses rekayasa fitur di tahap
berikutnya. Jika rasio cicilan terhadap pendapatan terbukti berkaitan
kuat dengan target, maka fitur tersebut menjadi kandidat utama dalam model.

### Pertanyaan Bisnis 2
**Seberapa baik riwayat perilaku pembayaran dan kondisi keuangan nasabah
dapat digunakan untuk memisahkan kelompok berisiko tinggi dari kelompok berisiko rendah?**

#### Konteks
Data historis tersedia dari beberapa sumber: catatan cicilan sebelumnya
melalui `repayment_history`, aktivitas rekening bulanan melalui `account_behavior`,
dan laporan biro kredit melalui `bureau_observations`.
Semua data ini harus digunakan dengan filter waktu yang ketat agar tidak
memasukkan informasi yang baru diketahui setelah keputusan pinjaman dibuat.

#### Yang Ingin Dijawab
Apakah nasabah yang pernah mengalami keterlambatan pembayaran sebelumnya
memiliki risiko yang lebih tinggi? Apakah nasabah dengan saldo rekening yang
sering rendah atau dengan pemanfaatan kredit yang tinggi cenderung masuk
ke kelompok target 1? Seberapa besar kontribusi data biro kredit dibandingkan
data rekening internal?

#### Relevansi Bisnis
Pertanyaan ini menyentuh inti dari penilaian kredit tradisional. Jika data
perilaku historis terbukti diskriminatif antara dua kelompok target,
maka model yang kita bangun memiliki fondasi yang kuat dan dapat
dipertanggungjawabkan secara bisnis.

#### Implikasi untuk Model
Jawaban dari pertanyaan ini akan menentukan apakah agregasi dari tabel
riwayat pembayaran, perilaku rekening, dan biro kredit layak dimasukkan ke model,
serta fitur agregat mana yang paling informatif.
Pengujian ini dilakukan pada tahap EDA dan rekayasa fitur.

### Pertanyaan Bisnis 3
**Bagaimana strategi validasi yang tepat agar estimasi performa model
pada data training dapat dipercaya sebagai indikator performa pada data test?**

#### Konteks
Dataset memiliki dimensi waktu. Pengajuan pinjaman terjadi pada berbagai tanggal
yang berbeda, dan setiap pengajuan memiliki cutoff waktu berupa `decision_datetime`
yang menentukan batas informasi yang boleh digunakan.
Selain itu, satu nasabah bisa memiliki beberapa pengajuan, sehingga ada
kemungkinan nasabah yang sama muncul di data training dan data validasi.

#### Yang Ingin Dijawab
Apakah pemisahan train dan validasi secara acak cukup, atau kita perlu melakukan
pemisahan berbasis waktu untuk menghindari kebocoran informasi?
Bagaimana menangani nasabah yang muncul di kedua sisi pemisahan data?
Apakah distribusi target pada data validasi mencerminkan distribusi pada data test?

#### Relevansi Bisnis
Strategi validasi yang buruk dapat membuat model terlihat bagus di lingkungan
pengembangan tetapi gagal saat dievaluasi pada data test kompetisi.
Ini adalah masalah bisnis nyata: sistem yang tampak akurat saat diuji
di laboratorium tetapi tidak bisa diandalkan saat diterapkan di lapangan
tidak bernilai bagi perusahaan.

#### Implikasi untuk Model
Jawaban dari pertanyaan ini akan menentukan arsitektur pipeline validasi
yang digunakan sepanjang proses pengembangan model. Keputusan ini harus
dibuat sebelum memulai eksperimen agar semua perbandingan model dilakukan
dengan cara yang konsisten dan dapat dipercaya.

## 3. Metrik Keberhasilan

Kompetisi ini menggunakan metrik resmi KS Statistic atau Kolmogorov Smirnov.
Di luar metrik kompetisi, kita juga mendefinisikan metrik pendukung
untuk memantau kualitas pipeline secara lebih menyeluruh.

### 3.1 KS Statistic (Metrik Utama)

KS Statistic mengukur perbedaan maksimum antara dua distribusi kumulatif:
distribusi skor prediksi untuk kelompok target 1 dan distribusi skor untuk
kelompok target 0.

Nilai KS berada di antara 0 dan 1. Semakin tinggi nilainya, semakin baik
model memisahkan kedua kelompok. Model yang menghasilkan skor acak akan
mendekati nilai KS nol.

Alasan tidak menggunakan akurasi adalah karena dataset memiliki ketidakseimbangan kelas.
Jika hanya sebagian kecil nasabah yang memiliki target 1, sebuah model yang selalu
memprediksi 0 akan memiliki akurasi tinggi tetapi gagal total dalam mengidentifikasi
nasabah berisiko. KS tidak terpengaruh oleh ketidakseimbangan kelas karena ia
mengevaluasi pemisahan distribusi skor, bukan akurasi prediksi pada satu ambang batas.

### 3.2 Metrik Pendukung

Selain KS, kita akan memantau metrik berikut selama proses pengembangan:

| Metrik | Fungsi |
|---|---|
| ROC AUC | Mengukur kemampuan pemeringkatan model secara keseluruhan |
| Gini Coefficient | Diturunkan dari AUC dengan rumus 2 kali AUC dikurangi 1 |
| Log Loss | Mengukur kualitas probabilitas yang dihasilkan model |
| PR AUC | Relevan untuk kelas tidak seimbang guna memantau ketepatan dan kelengkapan |

### 3.3 Standar Keberhasilan Pipeline

Di luar metrik numerik, pipeline dianggap berhasil jika memenuhi kriteria berikut:

| Kriteria | Standar |
|---|---|
| Tidak ada data leakage | Semua fitur dihitung hanya dari informasi yang tersedia sebelum decision datetime setiap pengajuan |
| Format submission valid | Setiap application id test muncul tepat satu kali dengan probabilitas antara 0 dan 1 |
| Reproduktifitas | Hasil model dapat direproduksi dengan acuan seed acak yang sama |
| Konsistensi validasi | Semua eksperimen menggunakan prosedur validasi yang seragam agar perbandingan adil |

## 4. Ringkasan

| Aspek | Penjelasan |
|---|---|
| Tujuan | Membangun model yang memprediksi probabilitas nasabah mengalami 90 DPD dalam 12 bulan |
| Input | Informasi pengajuan profil nasabah dan catatan riwayat historis |
| Output | Nilai default probability antara 0 dan 1 untuk setiap pengajuan di data test |
| Metrik utama | KS Statistic |
| Tantangan utama | Menggabungkan 8 tabel dengan aturan cutoff waktu yang ketat untuk menghindari data leakage |